# Probability pipeline
Compute the probability that each optical source is the true counterpart
of an X-ray source.

This function applies a scoring and probability framework combining
X-ray properties, optical information, and positional constraints.
Scores are converted into probabilities, weighted, normalized, and
adjusted with penalties to produce a final likelihood for each candidate.

Pipeline steps:
1) Initialize the scores of being AB, CV, LMXRB
2) Apply score based on the hardness classification, Mv vs X-ray, and optical classification
3) Apply weighting and normalization
4) Penalize candidates based on matching radius
5) Rescale probabilities across classes
6) Generate the final output table

In [2]:
import sys
import os
import numpy as np
import pandas as pd
sys.path.append(os.path.abspath(".."))

from src.optical_pipeline.classify_optical import optical_classification
from src.xray_pipeline.classify_xray import x_ray_classification
from src.find_candidates_pipeline.find_candidates import run_crossmatch_pipeline
from src.load_cluster_data import load_cluster_config

## Cluster (You can modify this)

In [3]:
cluster = "NGC_6809"

## Loading data

I run the optical classification, x-ray classification and find_counterpart pipeline.

In [4]:
params = load_cluster_config(cluster)

df_optical = optical_classification(distance_parsecs = params["distance_parsecs"], 
                                    path_optical = params["path_optical"],
                                    cluster_name = cluster)

df_xray = x_ray_classification(
    path_xray=params["path_xray"],
    BS_RA = params["BS_RA"],
    BS_Decl = params["BS_Decl"],
    distance_parsecs=params["distance_parsecs"],
    cluster_name = cluster
)

df = run_crossmatch_pipeline(df_optical, df_xray)
df



Optical classification completed


X-ray classification completed


Candidate search completed


,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6


## Highlight rows that are posible counterparts

In [5]:
def highlight_classification(row):
    if row["P_counterpart (%)"] >= 75:
        return ["background-color: blue"] * len(row)
    return [""] * len(row)

## Step 1: Assign Scores

### 1.1) initialize_scores
Initialize the classification score columns for each source.

This function creates score containers for the main classes (**AB**, **CV**, **LMXRB**),
setting all initial values to zero before applying scoring rules.

In [6]:
df["AB"] = 0.0
df["CV"] = 0.0
df["LMXRB"] = 0.0
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.0,0.0,0.0
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.0,0.0,0.0
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.0,0.0,0.0
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.0,0.0,0.0
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.0,0.0,0.0
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.0,0.0,0.0
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.0,0.0,0.0
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.0,0.0,0.0
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.0,0.0,0.0
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.0,0.0,0.0


### 1.2) apply_hardness_score
Assign scores based on X-ray hardness classification.

Each source receives a contribution to its class scores depending on its
hardness-based classification.

In [7]:
HC_score = 0.15
for i, row in df.iterrows():

    HC = row["Hardness_classification"]

    if HC == "LMXRB":
        df.loc[i, "LMXRB"] += HC_score

    elif HC == "CV":
        df.loc[i, "CV"] += HC_score

    elif HC == "CV & AB":
        df.loc[i, "CV"] += HC_score / 2
        df.loc[i, "AB"] += HC_score / 2

    elif HC == "AB":
        df.loc[i, "AB"] += HC_score
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.075,0.075,0.0
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.075,0.075,0.0
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.075,0.0
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.075,0.0
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.075,0.075,0.0
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.075,0.0
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.075,0.075,0.0
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.075,0.075,0.0
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.075,0.075,0.0
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.075,0.075,0.0


### 1.3) apply_mv_xray_score
Assign scores based on the **Mv-Lx** (optical magnitude vs X-ray luminosity) relation.

This step updates class scores using the classification derived from the
**Mv vs X-ray** relation.

In [8]:
MV_XR_score = 0.15
for i, row in df.iterrows():

    MV_XR = row["Mv vs xray"]

    if MV_XR == "LMXRB":
        df.loc[i, "LMXRB"] += MV_XR_score

    elif MV_XR == "CV":
        df.loc[i, "CV"] += MV_XR_score

    elif MV_XR == "AB":
        df.loc[i, "AB"] += MV_XR_score
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075,0.0
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075,0.0
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.225,0.0
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225,0.0
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.075,0.0
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225,0.0
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.075,0.0
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075,0.0
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075,0.0
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075,0.0


## 1.4) apply_optical_score
This function evaluates the location of each source relative to the Main
Sequence across different CMD configurations and updates class scores
accordingly. The scoring strategy is designed to reinforce the most
physically consistent classification while avoiding ambiguity so we 
get a secure classification.

Scoring rules
1) Blue side:
Sources located bluer than the Main Sequence are more likely to be Cataclysmic Variables (CVs) or Low-Mass X-ray Binaries (LMXRBs).
The score is assigned to the most probable class based on the previous classifications (X-ray properties and the Mv–Lx relation).
Additionally, sources that appear bluer in bluer CMD bands (**275**, **336**, **438**) are given higher weight, as they have a greater likelihood of being true counterparts compared to those identified in redder bands (**606**, **814**).
2) Red side:
Sources located redder than the Main Sequence are more likely to be Active Binaries (ABs).
A stronger contribution is applied if AB is already the dominant class; otherwise, a smaller score is assigned.
Similarly, the confidence increases when the classification is across redder bands CMDs than bluer bands.

3) Main Sequence:
Sources on the MS or MSTO slightly favor AB classification,
but only if AB is already the most probable class, to avoid
artificially inflating ambiguous cases.

In [9]:
optical_score = 0.7
for i, row in df.iterrows():

    for CMD_config in range(3):

        pos = row[f"pos_{CMD_config}"]

        # --- BLUE SIDE ---
        if pos in ["bluer than MS L1", "bluer than SGB L1", "bluer than MSTO L1"]:

            if row["LMXRB"] > row["CV"] and row["LMXRB"] > row["AB"]:
                df.loc[i, "LMXRB"] += (optical_score - (optical_score/6)*(CMD_config-1)) / 3

            elif row["CV"] > row["AB"]:
                df.loc[i, "CV"] += (optical_score - (optical_score/6)*(CMD_config-1)) / 3

            else:
                df.loc[i, "LMXRB"] += optical_score / 6
                df.loc[i, "CV"] += optical_score / 6

        # --- RED SIDE ---
        elif pos in ["redder than MS L1", "sub-sub-giant branch", "Red giant branch"]:

            if row["AB"] > row["CV"] and row["AB"] > row["LMXRB"]:
                df.loc[i, "AB"] += (optical_score + (optical_score/6)*(CMD_config-1)) / 3
            else:
                df.loc[i, "AB"] += (optical_score + (optical_score/6)*(CMD_config-1)) / 6

df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.497222,0.000000
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225000,0.000000
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.191667,0.116667
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225000,0.000000
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.425000,0.350000
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075000,0.000000
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000


## Step 2: Get probabilities

### 2.1) compute_base_probability
Compute the initial probability of being a true counterpart.

This function combines the class scores (AB, CV, LMXRB) into a total
counterpart probability for each candidate.

It computes the total probability for each x-ray source by substracting to 1
the probability that none of its candidates are the true counterpart.

In [10]:
# Calculating the probability for not being a counterpart
df["p_counterpart"] = df["AB"] + df["CV"] + df["LMXRB"]
df["p_not"] = 1 - df["p_counterpart"]

# The total probability for each x-ray source is 1 minus the probability that none of its candidates are the true counterpart
total_prob = 1 - df.groupby("Xray source")["p_not"].prod()
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB,p_counterpart,p_not
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.497222,0.000000,0.572222,4.277778e-01
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225000,0.000000,0.300000,7.000000e-01
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.191667,0.116667,0.533333,4.666667e-01
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225000,0.000000,0.300000,7.000000e-01
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.425000,0.350000,1.000000,1.110223e-16
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01


### 2.2) apply_weighting
Enhance probabilities based on classification confidence.

This step increases the weight of candidates with a clear dominant class.
The weighting is based on the ratio between the highest and second-highest
class scores, favoring sources with less ambiguous classifications.

In [11]:
weights = []

for _, row in df.iterrows():
    
    # For each row we create an array with the scores of being AB, CV, LMXRB and we sort it in descending order.
    arr = np.array([row["AB"], row["CV"], row["LMXRB"]])
    arr_sorted = np.sort(arr)[::-1]
    
    # We compute the ratio between the highest and second-highest score.
    if arr_sorted[1] != 0:
        ratio = arr_sorted[0] / arr_sorted[1]
    else:
        ratio = 1e9
    
    #We weight the probability of being a counterpart by this ratio, so that sources with a clear dominant class receive a higher probability.
    weights.append(row["p_counterpart"] * ratio)

df["p_weighted"] = weights
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB,p_counterpart,p_not,p_weighted
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.497222,0.000000,0.572222,4.277778e-01,3.793621
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.191667,0.116667,0.533333,4.666667e-01,0.626087
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.425000,0.350000,1.000000,1.110223e-16,1.214286
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000


### 2.3) normalize_probabilities
Normalize probabilities within each X-ray source.

The weighted probabilities are rescaled so that the total probability
distribution per X-ray source is consistent with the total_prob value
found in compute_base_probability.

In [12]:
# Calculating the total weighted probability for each x-ray source
sum_after = df.groupby("Xray source")["p_weighted"].sum()

new_probs = []

# For each candidate, we normalize the probability so the sum of the probabilities of all 
# candidates for a given x-ray source equals the total probability of that x-ray source.
for i, row in df.iterrows():

    x_id = row["Xray source"]

    p = row["p_weighted"] * total_prob.loc[x_id] / sum_after.loc[x_id]
    new_probs.append(p)

df["p_final"] = new_probs
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB,p_counterpart,p_not,p_weighted,p_final
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.118261
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.118261
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.497222,0.000000,0.572222,4.277778e-01,3.793621,0.498488
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000,0.118261
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.191667,0.116667,0.533333,4.666667e-01,0.626087,0.276239
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000,0.397094
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.425000,0.350000,1.000000,1.110223e-16,1.214286,0.212500
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.157500
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.157500
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.157500


### 2.4) apply_radius_penalty
Penalize candidates with less reliable positional matches.

Candidates that are not identified in the 95% confidence radius and
are identified using the fallback search radius (2 arcsec)
receive a reduction in their final probability.

In [13]:
df.loc[df["radius"] == "2", "p_final"] *= 0.8
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB,p_counterpart,p_not,p_weighted,p_final
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.094609
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.094609
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.075,0.497222,0.000000,0.572222,4.277778e-01,3.793621,0.398790
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000,0.094609
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.225,0.191667,0.116667,0.533333,4.666667e-01,0.626087,0.220991
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.075,0.225000,0.000000,0.300000,7.000000e-01,0.900000,0.317675
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.225,0.425000,0.350000,1.000000,1.110223e-16,1.214286,0.170000
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.126000
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.126000
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.225,0.075000,0.000000,0.300000,7.000000e-01,0.900000,0.126000


### 2.5) rescale_classes
Rescale class probabilities to match final counterpart likelihood.

After computing the final probability for each candidate, the class
scores (AB, CV, LMXRB) are proportionally adjusted so that their sum
remains consistent with the updated counterpart probability.

In [14]:
scale = df["p_final"] / df["p_counterpart"]

df["AB"] *= scale
df["CV"] *= scale
df["LMXRB"] *= scale
df

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,AB,CV,LMXRB,p_counterpart,p_not,p_weighted,p_final
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,0.070957,0.023652,0.000000,0.300000,7.000000e-01,0.900000,0.094609
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,0.070957,0.023652,0.000000,0.300000,7.000000e-01,0.900000,0.094609
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,0.052269,0.346522,0.000000,0.572222,4.277778e-01,3.793621,0.398790
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,0.023652,0.070957,0.000000,0.300000,7.000000e-01,0.900000,0.094609
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,0.093231,0.079419,0.048342,0.533333,4.666667e-01,0.626087,0.220991
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,0.079419,0.238256,0.000000,0.300000,7.000000e-01,0.900000,0.317675
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,0.038250,0.072250,0.059500,1.000000,1.110223e-16,1.214286,0.170000
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,0.094500,0.031500,0.000000,0.300000,7.000000e-01,0.900000,0.126000
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,0.094500,0.031500,0.000000,0.300000,7.000000e-01,0.900000,0.126000
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,0.094500,0.031500,0.000000,0.300000,7.000000e-01,0.900000,0.126000


## Step 3: finalize_output
Format and summarize the final probability results for each candidate.

This function aggregates probabilities per X-ray source and converts
all relevant quantities into percentage form for easier interpretation.
It prepares a clean output table containing the final class probabilities,
the counterpart likelihood, and the total probability associated with
each X-ray source.


A source is likely to be a counterpart if it has more than 80% on P_counterpart 

In [15]:
prob_sum = df.groupby("Xray source")["p_final"].sum()

df["total_prob"] = df["Xray source"].map(prob_sum)

df["Prob_AB (%)"] = (df["AB"] * 100).round(1)
df["Prob_CV (%)"] = (df["CV"] * 100).round(1)
df["Prob_LMXRB (%)"] = (df["LMXRB"] * 100).round(1)
df["P_counterpart (%)"] = (df["p_final"] * 100).round(1)
df["P_Xray_source (%)"] = (df["total_prob"] * 100).round(1)

df_final = df[["id_candidate", "Xray source", "opt_id", "Hardness_classification", "Mv vs xray", "pos_0", "pos_1", "pos_2", "radius",
               "n_sources", "Prob_AB (%)", "Prob_CV (%)", "Prob_LMXRB (%)",  "P_Xray_source (%)", "P_counterpart (%)"]]

df_final.style.apply(highlight_classification, axis=1)

,id_candidate,Xray source,opt_id,Hardness_classification,Mv vs xray,pos_0,pos_1,pos_2,radius,n_sources,Prob_AB (%),Prob_CV (%),Prob_LMXRB (%),P_Xray_source (%),P_counterpart (%)
0,CX2_1,CX2,R0044635,CV & AB,AB,MS,MS,MS,2,4,7.100000,2.400000,0.000000,68.300000,9.500000
1,CX2_2,CX2,R0044636,CV & AB,AB,MS,MS,MS,2,4,7.100000,2.400000,0.000000,68.300000,9.500000
2,CX2_3,CX2,R0044637,CV & AB,CV,bluer than MS L1,MS,MS,2,4,5.200000,34.700000,0.000000,68.300000,39.900000
3,CX2_4,CX2,R0045252,CV & AB,CV,MS,MS,MS,2,4,2.400000,7.100000,0.000000,68.300000,9.500000
4,CX8_1,CX8,R0000339,CV & AB,AB,bluer than SGB L1,RGB,RGB,2,2,9.300000,7.900000,4.800000,53.900000,22.100000
5,CX8_2,CX8,R0006914,CV & AB,CV,below the MS,MS,MS,2,2,7.900000,23.800000,0.000000,53.900000,31.800000
6,CX9_1,CX9,R0002365,CV & AB,AB,bluer than MSTO L1,bluer than MSTO L1,bluer than MSTO L1,2,6,3.800000,7.200000,5.900000,80.000000,17.000000
7,CX9_2,CX9,R0002374,CV & AB,AB,MSTO,MSTO,SGB,2,6,9.400000,3.200000,0.000000,80.000000,12.600000
8,CX9_3,CX9,R0031224,CV & AB,AB,MS,MS,MS,2,6,9.400000,3.200000,0.000000,80.000000,12.600000
9,CX9_4,CX9,R0032091,CV & AB,AB,MS,MS,MS,2,6,9.400000,3.200000,0.000000,80.000000,12.600000


## !A source is likely to be a counterpart if it has more than **75%** on **P_counterpart (%)**¡